In [ ]:
import numpy as np 
import pandas as pd 
import os

In [ ]:
import cv2
import matplotlib.pyplot as plt

image_dir = '/kaggle/input/camus-echocardiography-image-dataset/frames/'
mask_dir = '/kaggle/input/camus-echocardiography-image-dataset/masks/'

if not os.path.exists(image_dir):
    raise FileNotFoundError(f"Image directory {image_dir} does not exist")
if not os.path.exists(mask_dir):
    raise FileNotFoundError(f"Mask directory {mask_dir} does not exist")

image_files = [f for f in os.listdir(image_dir) if f.endswith('.png')]
mask_files = [f for f in os.listdir(mask_dir) if f.endswith('.png')]

print(f"Found {len(image_files)} image files in {image_dir}")
print(f"Found {len(mask_files)} mask files in {mask_dir}")

image_basenames = {os.path.splitext(f)[0].replace('frame_', ''): f for f in image_files}
mask_basenames = {os.path.splitext(f)[0].replace('mask_', ''): f for f in mask_files}

print("Sample image basenames:", list(image_basenames.keys())[:5])
print("Sample mask basenames:", list(mask_basenames.keys())[:5])

common_keys = set(image_basenames.keys()) & set(mask_basenames.keys())

print(f"Found {len(common_keys)} matching image-mask pairs")

data = []
for key in sorted(common_keys):
    image_path = os.path.join(image_dir, image_basenames[key])
    mask_path = os.path.join(mask_dir, mask_basenames[key])
    data.append({'image_path': image_path, 'mask_path': mask_path})

df = pd.DataFrame(data)

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.duplicated().sum()

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def display_image_mask_pairs(df, num_display=5):

    num_display = min(num_display, len(df))
    if num_display == 0:
        print("No image-mask pairs to display.")
        return
    
    display_df = df.head(num_display)
    
    fig, axes = plt.subplots(2, num_display, figsize=(5 * num_display, 6))
    
    if num_display == 1:
        axes = [[axes[0]], [axes[1]]]
    
    for i in range(num_display):
        image = cv2.imread(display_df['image_path'].iloc[i], cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(display_df['mask_path'].iloc[i], cv2.IMREAD_GRAYSCALE)
        
        axes[0][i].imshow(image, cmap='gray')
        axes[0][i].set_title(f'Image {os.path.basename(display_df["image_path"].iloc[i])}')
        axes[0][i].axis('off')
        
        axes[1][i].imshow(mask, cmap='gray')
        axes[1][i].set_title(f'Mask {os.path.basename(display_df["mask_path"].iloc[i])}')
        axes[1][i].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
display_image_mask_pairs(df, num_display=5)

In [ ]:
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)

def iou(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    union = tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

def unet_model(input_shape=(256, 256, 1)):
    inputs = layers.Input(input_shape)
    
    c1 = layers.Conv2D(64, 3, activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(64, 3, activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    
    c2 = layers.Conv2D(128, 3, activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(128, 3, activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)
    
    c3 = layers.Conv2D(256, 3, activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(256, 3, activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)
    
    c4 = layers.Conv2D(512, 3, activation='relu', padding='same')(p3)
    c4 = layers.Conv2D(512, 3, activation='relu', padding='same')(c4)
    p4 = layers.MaxPooling2D((2, 2))(c4)
  
    c5 = layers.Conv2D(1024, 3, activation='relu', padding='same')(p4)
    c5 = layers.Conv2D(1024, 3, activation='relu', padding='same')(c5)
    
    u6 = layers.Conv2DTranspose(512, 2, strides=(2, 2), padding='same')(c5)
    u6 = layers.concatenate([u6, c4])
    c6 = layers.Conv2D(512, 3, activation='relu', padding='same')(u6)
    c6 = layers.Conv2D(512, 3, activation='relu', padding='same')(c6)
    
    u7 = layers.Conv2DTranspose(256, 2, strides=(2, 2), padding='same')(c6)
    u7 = layers.concatenate([u7, c3])
    c7 = layers.Conv2D(256, 3, activation='relu', padding='same')(u7)
    c7 = layers.Conv2D(256, 3, activation='relu', padding='same')(c7)
    
    u8 = layers.Conv2DTranspose(128, 2, strides=(2, 2), padding='same')(c7)
    u8 = layers.concatenate([u8, c2])
    c8 = layers.Conv2D(128, 3, activation='relu', padding='same')(u8)
    c8 = layers.Conv2D(128, 3, activation='relu', padding='same')(c8)
    
    u9 = layers.Conv2DTranspose(64, 2, strides=(2, 2), padding='same')(c8)
    u9 = layers.concatenate([u9, c1])
    c9 = layers.Conv2D(64, 3, activation='relu', padding='same')(u9)
    c9 = layers.Conv2D(64, 3, activation='relu', padding='same')(c9)
    
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(c9)
    
    model = models.Model(inputs, outputs)
    return model

def load_and_preprocess_data(image_dir, mask_dir, img_size=(256, 256)):
   
    image_files = [f for f in os.listdir(image_dir) if f.endswith('.png')]
    mask_files = [f for f in os.listdir(mask_dir) if f.endswith('.png')]
    
    print(f"Found {len(image_files)} image files in {image_dir}")
    print(f"Found {len(mask_files)} mask files in {mask_dir}")
    
    image_basenames = {os.path.splitext(f)[0].replace('frame_', ''): f for f in image_files}
    mask_basenames = {os.path.splitext(f)[0].replace('mask_', ''): f for f in mask_files}
    
    common_keys = set(image_basenames.keys()) & set(mask_basenames.keys())
    print(f"Found {len(common_keys)} matching image-mask pairs")
    
    images = []
    masks = []
    for key in sorted(common_keys):
        img_path = os.path.join(image_dir, image_basenames[key])
        mask_path = os.path.join(mask_dir, mask_basenames[key])
        
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        img = cv2.resize(img, img_size)
        mask = cv2.resize(mask, img_size)
        
        img = img.astype(np.float32) / 255.0
        mask = (mask > 0).astype(np.float32)
        
        images.append(img)
        masks.append(mask)
    
    images = np.array(images)[..., np.newaxis]
    masks = np.array(masks)[..., np.newaxis]
    
    return images, masks

def plot_training_history(history):
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))
    
    ax1.plot(history.history['loss'], label='Train Loss')
    ax1.plot(history.history['val_loss'], label='Val Loss')
    ax1.set_title('Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    
    ax2.plot(history.history['dice_coefficient'], label='Train Dice')
    ax2.plot(history.history['val_dice_coefficient'], label='Val Dice')
    ax2.set_title('Dice Coefficient')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Dice')
    ax2.legend()
    
    ax3.plot(history.history['iou'], label='Train IoU')
    ax3.plot(history.history['val_iou'], label='Val IoU')
    ax3.set_title('IoU')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('IoU')
    ax3.legend()
    
    plt.tight_layout()
    plt.show()

def display_predictions(model, x_test, y_test, num_display=5):
    num_display = min(num_display, len(x_test))
    if num_display == 0:
        print("No test samples to display.")
        return
    
    preds = model.predict(x_test[:num_display])
    preds = (preds > 0.5).astype(np.float32)  # Binarize predictions
    
    fig, axes = plt.subplots(3, num_display, figsize=(5 * num_display, 7))
    if num_display == 1:
        axes = [[axes[0]], [axes[1]], [axes[2]]]
    
    for i in range(num_display):
       
        axes[0][i].imshow(x_test[i, :, :, 0], cmap='gray')
        axes[0][i].set_title(f'Image {i+1}')
        axes[0][i].axis('off')
        
    
        axes[1][i].imshow(y_test[i, :, :, 0], cmap='gray')
        axes[1][i].set_title(f'Ground Truth {i+1}')
        axes[1][i].axis('off')
        

        axes[2][i].imshow(preds[i, :, :, 0], cmap='gray')
        axes[2][i].set_title(f'Prediction {i+1}')
        axes[2][i].axis('off')
    
    plt.tight_layout()
    plt.show()

image_dir = '/kaggle/input/camus-echocardiography-image-dataset/frames/'
mask_dir = '/kaggle/input/camus-echocardiography-image-dataset/masks/'

images, masks = load_and_preprocess_data(image_dir, mask_dir)

x_train, x_temp, y_train, y_temp = train_test_split(images, masks, test_size=0.3, random_state=42)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42)
print(f"Train: {len(x_train)}, Validation: {len(x_val)}, Test: {len(x_test)}")

model = unet_model()
model.compile(optimizer='adam', loss='binary_crossentropy', 
              metrics=[dice_coefficient, iou])

history = model.fit(x_train, y_train, validation_data=(x_val, y_val), 
                   epochs=20, batch_size=16, verbose=1)

test_scores = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Loss: {test_scores[0]:.4f}")
print(f"Test Dice Coefficient: {test_scores[1]:.4f}")
print(f"Test IoU: {test_scores[2]:.4f}")

plot_training_history(history)

display_predictions(model, x_test, y_test, num_display=5)

In [ ]:
def unet_plus_plus(input_shape=(256, 256, 1)):
    inputs = layers.Input(input_shape)
    
    c1 = layers.Conv2D(64, 3, activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(64, 3, activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    
    c2 = layers.Conv2D(128, 3, activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(128, 3, activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)
    
    c3 = layers.Conv2D(256, 3, activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(256, 3, activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)
    
    c4 = layers.Conv2D(512, 3, activation='relu', padding='same')(p3)
    c4 = layers.Conv2D(512, 3, activation='relu', padding='same')(c4)
    p4 = layers.MaxPooling2D((2, 2))(c4)
    
    c5 = layers.Conv2D(1024, 3, activation='relu', padding='same')(p4)
    c5 = layers.Conv2D(1024, 3, activation='relu', padding='same')(c5)
    
    u4_1 = layers.Conv2DTranspose(512, 2, strides=(2, 2), padding='same')(c5)
    u4_1 = layers.concatenate([u4_1, c4])
    c4_1 = layers.Conv2D(512, 3, activation='relu', padding='same')(u4_1)
    c4_1 = layers.Conv2D(512, 3, activation='relu', padding='same')(c4_1)
    
    u3_1 = layers.Conv2DTranspose(256, 2, strides=(2, 2), padding='same')(c4)
    u3_1 = layers.concatenate([u3_1, c3])
    c3_1 = layers.Conv2D(256, 3, activation='relu', padding='same')(u3_1)
    c3_1 = layers.Conv2D(256, 3, activation='relu', padding='same')(c3_1)
    
    u3_2 = layers.Conv2DTranspose(256, 2, strides=(2, 2), padding='same')(c4_1)
    u3_2 = layers.concatenate([u3_2, c3, c3_1])
    c3_2 = layers.Conv2D(256, 3, activation='relu', padding='same')(u3_2)
    c3_2 = layers.Conv2D(256, 3, activation='relu', padding='same')(c3_2)
    
    u2_1 = layers.Conv2DTranspose(128, 2, strides=(2, 2), padding='same')(c3)
    u2_1 = layers.concatenate([u2_1, c2])
    c2_1 = layers.Conv2D(128, 3, activation='relu', padding='same')(u2_1)
    c2_1 = layers.Conv2D(128, 3, activation='relu', padding='same')(c2_1)
    
    u2_2 = layers.Conv2DTranspose(128, 2, strides=(2, 2), padding='same')(c3_1)
    u2_2 = layers.concatenate([u2_2, c2, c2_1])
    c2_2 = layers.Conv2D(128, 3, activation='relu', padding='same')(u2_2)
    c2_2 = layers.Conv2D(128, 3, activation='relu', padding='same')(c2_2)
    
    u2_3 = layers.Conv2DTranspose(128, 2, strides=(2, 2), padding='same')(c3_2)
    u2_3 = layers.concatenate([u2_3, c2, c2_1, c2_2])
    c2_3 = layers.Conv2D(128, 3, activation='relu', padding='same')(u2_3)
    c2_3 = layers.Conv2D(128, 3, activation='relu', padding='same')(c2_3)
    
    u1_1 = layers.Conv2DTranspose(64, 2, strides=(2, 2), padding='same')(c2)
    u1_1 = layers.concatenate([u1_1, c1])
    c1_1 = layers.Conv2D(64, 3, activation='relu', padding='same')(u1_1)
    c1_1 = layers.Conv2D(64, 3, activation='relu', padding='same')(c1_1)
    
    u1_2 = layers.Conv2DTranspose(64, 2, strides=(2, 2), padding='same')(c2_1)
    u1_2 = layers.concatenate([u1_2, c1, c1_1])
    c1_2 = layers.Conv2D(64, 3, activation='relu', padding='same')(u1_2)
    c1_2 = layers.Conv2D(64, 3, activation='relu', padding='same')(c1_2)
    
    u1_3 = layers.Conv2DTranspose(64, 2, strides=(2, 2), padding='same')(c2_2)
    u1_3 = layers.concatenate([u1_3, c1, c1_1, c1_2])
    c1_3 = layers.Conv2D(64, 3, activation='relu', padding='same')(u1_3)
    c1_3 = layers.Conv2D(64, 3, activation='relu', padding='same')(c1_3)
    
    u1_4 = layers.Conv2DTranspose(64, 2, strides=(2, 2), padding='same')(c2_3)
    u1_4 = layers.concatenate([u1_4, c1, c1_1, c1_2, c1_3])
    c1_4 = layers.Conv2D(64, 3, activation='relu', padding='same')(u1_4)
    c1_4 = layers.Conv2D(64, 3, activation='relu', padding='same')(c1_4)
    
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(c1_4)
    
    model = models.Model(inputs, outputs)
    return model

def load_and_preprocess_data(image_dir, mask_dir, img_size=(256, 256)):
    image_files = [f for f in os.listdir(image_dir) if f.endswith('.png')]
    mask_files = [f for f in os.listdir(mask_dir) if f.endswith('.png')]
    print(f"Found {len(image_files)} image files in {image_dir}")
    print(f"Found {len(mask_files)} mask files in {mask_dir}")
    image_basenames = {os.path.splitext(f)[0].replace('frame_', ''): f for f in image_files}
    mask_basenames = {os.path.splitext(f)[0].replace('mask_', ''): f for f in mask_files}
    common_keys = set(image_basenames.keys()) & set(mask_basenames.keys())
    print(f"Found {len(common_keys)} matching image-mask pairs")
    
    images = []
    masks = []
    
    for key in sorted(common_keys):
        img_path = os.path.join(image_dir, image_basenames[key])
        mask_path = os.path.join(mask_dir, mask_basenames[key])
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, img_size)
        mask = cv2.resize(mask, img_size)
        img = img.astype(np.float32) / 255.0
        mask = (mask > 0).astype(np.float32)
        images.append(img)
        masks.append(mask)
    images = np.array(images)[..., np.newaxis]
    masks = np.array(masks)[..., np.newaxis]
    return images, masks

def plot_training_history(history):
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))
    ax1.plot(history.history['loss'], label='Train Loss')
    ax1.plot(history.history['val_loss'], label='Val Loss')
    ax1.set_title('Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    
    ax2.plot(history.history['dice_coefficient'], label='Train Dice')
    ax2.plot(history.history['val_dice_coefficient'], label='Val Dice')
    ax2.set_title('Dice Coefficient')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Dice')
    ax2.legend()
    
    ax3.plot(history.history['iou'], label='Train IoU')
    ax3.plot(history.history['val_iou'], label='Val IoU')
    ax3.set_title('IoU')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('IoU')
    ax3.legend()
    plt.tight_layout()
    plt.show()

def display_predictions(model, x_test, y_test, num_display=5):
    num_display = min(num_display, len(x_test))
    if num_display == 0:
        print("No test samples to display.")
        return
        
    preds = model.predict(x_test[:num_display])
    preds = (preds > 0.5).astype(np.float32)
    fig, axes = plt.subplots(3, num_display, figsize=(5 * num_display, 7))
    
    if num_display == 1:
        axes = [[axes[0]], [axes[1]], [axes[2]]]
        
    for i in range(num_display):
        axes[0][i].imshow(x_test[i, :, :, 0], cmap='gray')
        axes[0][i].set_title(f'Image {i+1}')
        axes[0][i].axis('off')
        axes[1][i].imshow(y_test[i, :, :, 0], cmap='gray')
        axes[1][i].set_title(f'Ground Truth {i+1}')
        axes[1][i].axis('off')
        axes[2][i].imshow(preds[i, :, :, 0], cmap='gray')
        axes[2][i].set_title(f'Prediction {i+1}')
        axes[2][i].axis('off')
    plt.tight_layout()
    plt.show()

image_dir = '/kaggle/input/camus-echocardiography-image-dataset/frames/'
mask_dir = '/kaggle/input/camus-echocardiography-image-dataset/masks/'

images, masks = load_and_preprocess_data(image_dir, mask_dir)

x_train, x_temp, y_train, y_temp = train_test_split(images, masks, test_size=0.3, random_state=42)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {len(x_train)}, Validation: {len(x_val)}, Test: {len(x_test)}")
model = unet_plus_plus()

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=[dice_coefficient, iou])

history = model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=20, batch_size=16, verbose=1)

test_scores = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Loss: {test_scores[0]:.4f}")
print(f"Test Dice Coefficient: {test_scores[1]:.4f}")
print(f"Test IoU: {test_scores[2]:.4f}")

plot_training_history(history)

display_predictions(model, x_test, y_test, num_display=5)

In [ ]:
def inception_block(inputs, filters):
    conv1x1 = layers.Conv2D(filters // 4, 1, activation='relu', padding='same')(inputs)
    
    conv3x3 = layers.Conv2D(filters // 4, 1, activation='relu', padding='same')(inputs)
    conv3x3 = layers.Conv2D(filters // 4, 3, activation='relu', padding='same')(conv3x3)
    
    conv5x5 = layers.Conv2D(filters // 4, 1, activation='relu', padding='same')(inputs)
    conv5x5 = layers.Conv2D(filters // 4, 5, activation='relu', padding='same')(conv5x5)
    
    pool = layers.MaxPooling2D(3, strides=1, padding='same')(inputs)
    pool = layers.Conv2D(filters // 4, 1, activation='relu', padding='same')(pool)
    
    output = layers.concatenate([conv1x1, conv3x3, conv5x5, pool])
    return output

def inception_unet_plus_plus(input_shape=(128, 128, 1)):
    inputs = layers.Input(input_shape)
    
    c1 = inception_block(inputs, 32)
    c1 = inception_block(c1, 32)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    
    c2 = inception_block(p1, 64)
    c2 = inception_block(c2, 64)
    p2 = layers.MaxPooling2D((2, 2))(c2)
    
    c3 = inception_block(p2, 128)
    c3 = inception_block(c3, 128)
    p3 = layers.MaxPooling2D((2, 2))(c3)
    
    c4 = inception_block(p3, 256)
    c4 = inception_block(c4, 256)
    p4 = layers.MaxPooling2D((2, 2))(c4)
    
    c5 = inception_block(p4, 512)
    c5 = inception_block(c5, 512)
    
    u4_1 = layers.Conv2DTranspose(256, 2, strides=(2, 2), padding='same')(c5)
    u4_1 = layers.concatenate([u4_1, c4])
    c4_1 = inception_block(u4_1, 256)
    c4_1 = inception_block(c4_1, 256)
    
    u3_1 = layers.Conv2DTranspose(128, 2, strides=(2, 2), padding='same')(c4)
    u3_1 = layers.concatenate([u3_1, c3])
    c3_1 = inception_block(u3_1, 128)
    c3_1 = inception_block(c3_1, 128)
    
    u3_2 = layers.Conv2DTranspose(128, 2, strides=(2, 2), padding='same')(c4_1)
    u3_2 = layers.concatenate([u3_2, c3, c3_1])
    c3_2 = inception_block(u3_2, 128)
    c3_2 = inception_block(c3_2, 128)
    
    u2_1 = layers.Conv2DTranspose(64, 2, strides=(2, 2), padding='same')(c3)
    u2_1 = layers.concatenate([u2_1, c2])
    c2_1 = inception_block(u2_1, 64)
    c2_1 = inception_block(c2_1, 64)
    
    u2_2 = layers.Conv2DTranspose(64, 2, strides=(2, 2), padding='same')(c3_1)
    u2_2 = layers.concatenate([u2_2, c2, c2_1])
    c2_2 = inception_block(u2_2, 64)
    c2_2 = inception_block(c2_2, 64)
    
    u2_3 = layers.Conv2DTranspose(64, 2, strides=(2, 2), padding='same')(c3_2)
    u2_3 = layers.concatenate([u2_3, c2, c2_1, c2_2])
    c2_3 = inception_block(u2_3, 64)
    c2_3 = inception_block(c2_3, 64)
    
    u1_1 = layers.Conv2DTranspose(32, 2, strides=(2, 2), padding='same')(c2)
    u1_1 = layers.concatenate([u1_1, c1])
    c1_1 = inception_block(u1_1, 32)
    c1_1 = inception_block(c1_1, 32)
    
    u1_2 = layers.Conv2DTranspose(32, 2, strides=(2, 2), padding='same')(c2_1)
    u1_2 = layers.concatenate([u1_2, c1, c1_1])
    c1_2 = inception_block(u1_2, 32)
    c1_2 = inception_block(c1_2, 32)
    
    u1_3 = layers.Conv2DTranspose(32, 2, strides=(2, 2), padding='same')(c2_2)
    u1_3 = layers.concatenate([u1_3, c1, c1_1, c1_2])
    c1_3 = inception_block(u1_3, 32)
    c1_3 = inception_block(c1_3, 32)
    
    u1_4 = layers.Conv2DTranspose(32, 2, strides=(2, 2), padding='same')(c2_3)
    u1_4 = layers.concatenate([u1_4, c1, c1_1, c1_2, c1_3])
    c1_4 = inception_block(u1_4, 32)
    c1_4 = inception_block(c1_4, 32)
    
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(c1_4)
    
    model = models.Model(inputs, outputs)
    return model

def load_and_preprocess_data(image_dir, mask_dir, img_size=(128, 128)):
    image_files = [f for f in os.listdir(image_dir) if f.endswith('.png')]
    mask_files = [f for f in os.listdir(mask_dir) if f.endswith('.png')]
    print(f"Found {len(image_files)} image files in {image_dir}")
    print(f"Found {len(mask_files)} mask files in {mask_dir}")
    image_basenames = {os.path.splitext(f)[0].replace('frame_', ''): f for f in image_files}
    mask_basenames = {os.path.splitext(f)[0].replace('mask_', ''): f for f in mask_files}
    common_keys = set(image_basenames.keys()) & set(mask_basenames.keys())
    print(f"Found {len(common_keys)} matching image-mask pairs")
    
    images = []
    masks = []
    
    for key in sorted(common_keys):
        img_path = os.path.join(image_dir, image_basenames[key])
        mask_path = os.path.join(mask_dir, mask_basenames[key])
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, img_size)
        mask = cv2.resize(mask, img_size)
        img = img.astype(np.float32) / 255.0
        mask = (mask > 0).astype(np.float32)
        images.append(img)
        masks.append(mask)
        
    images = np.array(images)[..., np.newaxis]
    masks = np.array(masks)[..., np.newaxis]
    return images, masks

def plot_training_history(history):
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))
    ax1.plot(history.history['loss'], label='Train Loss')
    ax1.plot(history.history['val_loss'], label='Val Loss')
    ax1.set_title('Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax2.plot(history.history['dice_coefficient'], label='Train Dice')
    ax2.plot(history.history['val_dice_coefficient'], label='Val Dice')
    ax2.set_title('Dice Coefficient')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Dice')
    ax2.legend()
    ax3.plot(history.history['iou'], label='Train IoU')
    ax3.plot(history.history['val_iou'], label='Val IoU')
    ax3.set_title('IoU')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('IoU')
    ax3.legend()
    plt.tight_layout()
    plt.show()

def display_predictions(model, x_test, y_test, num_display=5):
    num_display = min(num_display, len(x_test))
    if num_display == 0:
        print("No test samples to display.")
        return
    preds = model.predict(x_test[:num_display])
    preds = (preds > 0.5).astype(np.float32)
    fig, axes = plt.subplots(3, num_display, figsize=(5 * num_display, 7))
    
    if num_display == 1:
        axes = [[axes[0]], [axes[1]], [axes[2]]]
        
    for i in range(num_display):
        axes[0][i].imshow(x_test[i, :, :, 0], cmap='gray')
        axes[0][i].set_title(f'Image {i+1}')
        axes[0][i].axis('off')
        axes[1][i].imshow(y_test[i, :, :, 0], cmap='gray')
        axes[1][i].set_title(f'Ground Truth {i+1}')
        axes[1][i].axis('off')
        axes[2][i].imshow(preds[i, :, :, 0], cmap='gray')
        axes[2][i].set_title(f'Prediction {i+1}')
        axes[2][i].axis('off')
    plt.tight_layout()
    plt.show()

image_dir = '/kaggle/input/camus-echocardiography-image-dataset/frames/'
mask_dir = '/kaggle/input/camus-echocardiography-image-dataset/masks/'
images, masks = load_and_preprocess_data(image_dir, mask_dir)
x_train, x_temp, y_train, y_temp = train_test_split(images, masks, test_size=0.3, random_state=42)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42)
print(f"Train: {len(x_train)}, Validation: {len(x_val)}, Test: {len(x_test)}")
model = inception_unet_plus_plus()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=[dice_coefficient, iou])
history = model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=20, batch_size=8, verbose=1)
test_scores = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Loss: {test_scores[0]:.4f}")
print(f"Test Dice Coefficient: {test_scores[1]:.4f}")
print(f"Test IoU: {test_scores[2]:.4f}")
plot_training_history(history)
display_predictions(model, x_test, y_test, num_display=5)